# Tokenization for Language Models

**Prerequisites**

- L02: NLP introduction (unigram/bigram tokenization, normalization, bag of words)
- L08: Embeddings (`nn.Embedding`, learned dense representations)

**Outcomes**

- Understand why tokenization matters for neural language models
- Implement character-level and word-level tokenizers from scratch
- Understand the limitations of both extremes
- Learn the BPE algorithm and implement a minimal version
- Connect tokenization to the embedding layer that feeds a transformer

In [ ]:
import re
import collections

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

np.set_printoptions(linewidth=140, precision=4, suppress=True)

%matplotlib inline

We will work with a small corpus of economics text throughout this notebook. The passages below are drawn from Adam Smith's *The Wealth of Nations* (1776), which is in the public domain.

In [ ]:
corpus = (
    "The annual labour of every nation is the fund which originally supplies "
    "it with all the necessaries and conveniences of life which it annually "
    "consumes, and which consist either in the immediate produce of that "
    "labour, or in what is purchased with that produce from other nations. "
    "According therefore as this produce, or what is purchased with it, bears "
    "a greater or smaller proportion to the number of those who are to consume "
    "it, the nation will be better or worse supplied with all the necessaries "
    "and conveniences for which it has occasion. But this proportion must in "
    "every nation be regulated by two different circumstances; first, by the "
    "skill, dexterity, and judgment with which its labour is generally applied; "
    "and, secondly, by the proportion between the number of those who are "
    "employed in useful labour, and that of those who are not so employed. "
    "Whatever be the soil, climate, or extent of territory of any particular "
    "nation, the abundance or scantiness of its annual supply must, in that "
    "particular situation, depend upon those two circumstances. "
    "The greatest improvement in the productive powers of labour, and the "
    "greater part of the skill, dexterity, and judgment with which it is "
    "anywhere directed, or applied, seem to have been the effects of the "
    "division of labour. The effects of the division of labour, in the general "
    "business of society, will be more easily understood by considering in "
    "what manner it operates in some particular manufactures. "
    "To take an example, therefore, from a very trifling manufacture; but one "
    "in which the division of labour has been very often taken notice of, the "
    "trade of the pin-maker; a workman not educated to this business, nor "
    "acquainted with the use of the machinery employed in it, could scarce, "
    "perhaps, with his utmost industry, make one pin in a day, and certainly "
    "could not make twenty. But in the way in which this business is now "
    "carried on, not only the whole work is a peculiar trade, but it is "
    "divided into a number of branches, of which the greater part are likewise "
    "a peculiar trade. One man draws out the wire, another straights it, a "
    "third cuts it, a fourth points it, a fifth grinds it at the top for "
    "receiving the head; to make the head requires two or three distinct "
    "operations; to put it on is a peculiar business, to whiten the pins is "
    "another; it is even a trade by itself to put them into the paper; and the "
    "important business of making a pin is, in this manner, divided into about "
    "eighteen distinct operations, which, in some manufactories, are all "
    "performed by distinct hands, though in others the same man will sometimes "
    "perform two or three of them. I have seen a small manufactory of this "
    "kind where ten men only were employed, and where some of them consequently "
    "performed two or three distinct operations. But though they were very poor, "
    "and therefore but indifferently accommodated with the necessary machinery, "
    "they could, when they exerted themselves, make among them about twelve "
    "pounds of pins in a day. There are in a pound upwards of four thousand "
    "pins of a middling size. Those ten persons, therefore, could make among "
    "them upwards of forty-eight thousand pins in a day."
)

print(f"Corpus length: {len(corpus):,} characters")
print(f"Corpus words:  {len(corpus.split()):,}")

## From Bag of Words to Sequences

In L02 we converted text into numeric data using **bag of words** and **binary presence**. Both approaches produce a single fixed-length vector for each document — a count (or indicator) for each word in the vocabulary. This is simple and often effective for classification, but it throws away a critical piece of information: **word order**.

Consider two sentences:

> "The Fed raised rates because inflation was high."
>
> "Inflation was high because the Fed raised rates."

A bag of words representation treats these identically. Yet they describe different causal stories — the first says the Fed *responded to* inflation, the second says the Fed *caused* inflation. For the models we are about to study (transformers), preserving order is essential.

Instead of mapping an entire document to one vector, we need a mapping from text to a **sequence of integers** — one integer per token. Each integer is an index into a vocabulary, and the order of the integers preserves the order of the original text.

A **tokenizer** implements this mapping. Formally, it defines:

1. A **vocabulary** $V = \{v_1, v_2, \ldots, v_{|V|}\}$ — the set of all recognized text segments
2. An **encode** function $\text{encode}: \text{string} \to \mathbb{N}^T$ — maps text to a sequence of token IDs
3. A **decode** function $\text{decode}: \mathbb{N}^T \to \text{string}$ — maps token IDs back to text

The central design question is: **what should a token be?**

## The Tokenization Spectrum

There is a fundamental trade-off between the **size of the vocabulary** and the **length of the encoded sequence**. At one extreme, we can make each character a token (tiny vocabulary, long sequences). At the other extreme, each word is a token (huge vocabulary, short sequences). Subword methods sit in between.

| Strategy | Vocabulary size | Sequence length | OOV problem? |
|---|---|---|---|
| Character | ~100 | Very long | No |
| Subword (BPE) | 30k–100k | Medium | No |
| Word | 100k+ | Short | Yes |

The **out-of-vocabulary (OOV) problem** arises when a tokenizer encounters a word it has never seen. Character-level and subword tokenizers can always fall back to smaller units, but a word-level tokenizer must either discard the unknown word or replace it with a special `<UNK>` token.

We will implement all three strategies and compare them.

## Character-Level Tokenization

The simplest tokenizer treats each character as a separate token. The vocabulary is just the set of unique characters in the training corpus.

**Advantages**:
- Very small vocabulary (typically < 256 for English text)
- No out-of-vocabulary problem — any string can be encoded
- No preprocessing required (no need for stemming, lemmatization, etc.)

**Disadvantages**:
- Very long sequences — each word becomes many tokens
- Each token carries little semantic meaning on its own ("q" tells us much less than "quantitative")
- Models must learn to compose characters into meaningful units, which requires more data and compute

In [ ]:
class CharTokenizer:
    """Character-level tokenizer."""

    def __init__(self, text):
        chars = sorted(set(text))
        self.char_to_id = {ch: i for i, ch in enumerate(chars)}
        self.id_to_char = {i: ch for ch, i in self.char_to_id.items()}
        self.vocab_size = len(chars)

    def encode(self, text):
        return [self.char_to_id[ch] for ch in text]

    def decode(self, ids):
        return "".join(self.id_to_char[i] for i in ids)


char_tok = CharTokenizer(corpus)
print(f"Vocabulary size: {char_tok.vocab_size}")
print(f"Vocabulary: {sorted(char_tok.char_to_id.keys())}")

In [ ]:
sample = "the division of labour"
encoded = char_tok.encode(sample)

print(f"Original:  '{sample}'")
print(f"Encoded:   {encoded}")
print(f"Decoded:   '{char_tok.decode(encoded)}'")
print(f"Tokens:    {len(encoded)}")

A 22-character string becomes a sequence of 22 tokens. The word "labour" alone requires 6 tokens, each of which could just as easily belong to any other word.

## Word-Level Tokenization

This is the approach we used in L02: split the text on whitespace and punctuation, and assign each unique word an integer ID.

**Advantages**:
- Each token is a semantically meaningful unit
- Short sequences (one token per word)

**Disadvantages**:
- Very large vocabulary, especially with morphologically rich languages
- Cannot handle words not seen during training (the OOV problem)
- Different forms of the same word ("run", "runs", "running") are entirely separate tokens with no shared structure

In [ ]:
class WordTokenizer:
    """Word-level tokenizer with simple whitespace + punctuation splitting."""

    def __init__(self, text):
        # Split on whitespace, then separate punctuation
        tokens = re.findall(r"\w+|[^\w\s]", text.lower())
        vocab = sorted(set(tokens))
        self.token_to_id = {tok: i for i, tok in enumerate(vocab)}
        self.id_to_token = {i: tok for tok, i in self.token_to_id.items()}
        self.vocab_size = len(vocab)

    def encode(self, text):
        tokens = re.findall(r"\w+|[^\w\s]", text.lower())
        out = []
        for tok in tokens:
            if tok in self.token_to_id:
                out.append(self.token_to_id[tok])
            else:
                out.append(-1)  # OOV marker
        return out

    def decode(self, ids):
        return " ".join(self.id_to_token.get(i, "<UNK>") for i in ids)


word_tok = WordTokenizer(corpus)
print(f"Vocabulary size: {word_tok.vocab_size}")

In [ ]:
sample = "the division of labour"
encoded = word_tok.encode(sample)

print(f"Original:  '{sample}'")
print(f"Encoded:   {encoded}")
print(f"Decoded:   '{word_tok.decode(encoded)}'")
print(f"Tokens:    {len(encoded)}")

Four tokens instead of 22 — much more compact. But what happens when we encounter a word that wasn't in Adam Smith's vocabulary?

In [ ]:
# OOV demonstration
oos = "quantitative easing affects inflation expectations"
encoded_oos = word_tok.encode(oos)

print(f"Input:   '{oos}'")
print(f"Encoded: {encoded_oos}")
print(f"Decoded: '{word_tok.decode(encoded_oos)}'")

Most of the words are unknown — the tokenizer has lost nearly all of the information.

### Comparing the two extremes

In [ ]:
# Encode the full corpus with both tokenizers
char_encoded = char_tok.encode(corpus)
word_encoded = word_tok.encode(corpus)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Vocabulary size
axes[0].bar(["Character", "Word"], [char_tok.vocab_size, word_tok.vocab_size],
            color=["steelblue", "darkorange"], edgecolor="black", linewidth=0.5)
for i, v in enumerate([char_tok.vocab_size, word_tok.vocab_size]):
    axes[0].text(i, v + 3, str(v), ha="center", fontsize=11)
axes[0].set_ylabel("Vocabulary size")
axes[0].set_title("Vocabulary size")

# Sequence length
axes[1].bar(["Character", "Word"], [len(char_encoded), len(word_encoded)],
            color=["steelblue", "darkorange"], edgecolor="black", linewidth=0.5)
for i, v in enumerate([len(char_encoded), len(word_encoded)]):
    axes[1].text(i, v + 10, str(v), ha="center", fontsize=11)
axes[1].set_ylabel("Sequence length (tokens)")
axes[1].set_title("Sequence length for full corpus")

plt.tight_layout()
plt.show()

Character tokenization gives us a tiny vocabulary but very long sequences. Word tokenization gives us short sequences but a large vocabulary that cannot handle new words. We want something in between.

## Subword Tokenization: Byte Pair Encoding

**Byte Pair Encoding (BPE)** was originally a data compression algorithm (Gage, 1994) that was adapted for NLP tokenization by Sennrich et al. (2016). It is the tokenization method behind GPT-2, GPT-4, and LLaMA.

The core idea is simple: start with individual characters, then **iteratively merge the most frequent adjacent pair** into a new token. After $M$ merges, the vocabulary has grown from the base character set to include common character sequences — these are our subword tokens.

### The algorithm

Given a training corpus:

1. Initialize the vocabulary $V$ as the set of all individual characters in the corpus
2. Represent the corpus as a sequence of characters (the initial tokenization)
3. For $i = 1, \ldots, M$ (number of merges):
   - Count all adjacent token pairs $(v_a, v_b)$ in the current tokenization
   - Find the most frequent pair: $(v_a^*, v_b^*) = \arg\max_{(v_a, v_b)} \text{count}(v_a, v_b)$
   - Create a new token $v_{\text{new}} = v_a^* \| v_b^*$ (string concatenation)
   - Add $v_{\text{new}}$ to $V$
   - Replace every occurrence of $(v_a^*, v_b^*)$ in the corpus with $v_{\text{new}}$
4. Return $V$ and the ordered list of merge rules

The number of merges $M$ is a hyperparameter that controls the vocabulary size: $|V| = |\text{base characters}| + M$.

In [ ]:
def get_pair_counts(token_lists):
    """Count adjacent token pairs across all words."""
    counts = collections.Counter()
    for tokens in token_lists:
        for i in range(len(tokens) - 1):
            counts[(tokens[i], tokens[i + 1])] += 1
    return counts


def merge_pair(token_lists, pair):
    """Replace all occurrences of `pair` with the merged token."""
    merged = pair[0] + pair[1]
    new_lists = []
    for tokens in token_lists:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(merged)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        new_lists.append(new_tokens)
    return new_lists


def train_bpe(text, num_merges):
    """
    Train BPE on a text corpus.

    Returns the merge rules and final vocabulary.
    """
    # Split into words, keep spaces as part of the next word (GPT-2 convention)
    words = re.findall(r"\S+|\s", text)
    # Initialize: each word is a list of characters
    token_lists = [list(w) for w in words]

    merges = []
    for i in range(num_merges):
        counts = get_pair_counts(token_lists)
        if not counts:
            break
        best_pair = max(counts, key=counts.get)
        token_lists = merge_pair(token_lists, best_pair)
        merges.append(best_pair)

    # Build vocabulary from all tokens that appear
    vocab = sorted(set(tok for tokens in token_lists for tok in tokens))

    return merges, vocab, token_lists

In [ ]:
merges, vocab, token_lists = train_bpe(corpus, num_merges=200)

print(f"Vocabulary size after 200 merges: {len(vocab)}")
print(f"\nFirst 20 merge rules:")
for i, (a, b) in enumerate(merges[:20]):
    print(f"  {i+1:3d}. '{a}' + '{b}' -> '{a+b}'")

The first merges combine the most common character pairs — `t` + `h` → `th`, `th` + `e` → `the`, etc. As training proceeds, the merges create increasingly long and semantically meaningful tokens.

In [ ]:
# Show what the tokenized corpus looks like
flat_tokens = [tok for tokens in token_lists for tok in tokens]

print(f"Total tokens in corpus: {len(flat_tokens)}")
print(f"\nFirst 40 tokens:")
print(flat_tokens[:40])

Now let's build an encoder that applies the learned merge rules to new text.

In [ ]:
def bpe_encode(text, merges):
    """Encode text using learned BPE merge rules."""
    words = re.findall(r"\S+|\s", text)
    token_lists = [list(w) for w in words]

    # Apply merges in the order they were learned
    for pair in merges:
        token_lists = merge_pair(token_lists, pair)

    return [tok for tokens in token_lists for tok in tokens]


# Encode our earlier test sentences
sample = "the division of labour"
bpe_tokens = bpe_encode(sample, merges)
print(f"Input:  '{sample}'")
print(f"Tokens: {bpe_tokens}")
print(f"Count:  {len(bpe_tokens)}")

In [ ]:
# The OOV sentence that broke word tokenization
oos = "quantitative easing affects inflation expectations"
bpe_tokens_oos = bpe_encode(oos, merges)
print(f"Input:  '{oos}'")
print(f"Tokens: {bpe_tokens_oos}")
print(f"Count:  {len(bpe_tokens_oos)}")

BPE handles the unseen sentence gracefully. Common substrings like "tion" and "at" are recognized as single tokens, while rarer character sequences are broken into smaller pieces. No information is lost.

### How vocabulary size and sequence length change with merges

In [ ]:
merge_counts = list(range(0, 301, 10))
vocab_sizes = []
seq_lengths = []

for m in merge_counts:
    _, v, tl = train_bpe(corpus, num_merges=m)
    vocab_sizes.append(len(v))
    seq_lengths.append(sum(len(t) for t in tl))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(merge_counts, vocab_sizes, color="steelblue", marker="o", markersize=3)
axes[0].set_xlabel("Number of merges")
axes[0].set_ylabel("Vocabulary size")
axes[0].set_title("Vocabulary size vs. merges")

axes[1].plot(merge_counts, seq_lengths, color="darkorange", marker="o", markersize=3)
axes[1].set_xlabel("Number of merges")
axes[1].set_ylabel("Total tokens in corpus")
axes[1].set_title("Sequence length vs. merges")

plt.tight_layout()
plt.show()

As we perform more merges, the vocabulary grows linearly while the total number of tokens decreases — each merge replaces two tokens with one. The sequence length drops steeply at first (merging very common pairs) and then more slowly.

In practice, GPT-2 uses ~50,000 merges, GPT-4 uses ~100,000, and LLaMA uses ~32,000. The right number depends on the language, the corpus, and the downstream model architecture.

### BPE in Practice

Production tokenizers use the same algorithm we implemented above, but with several refinements:

- **Byte-level BPE**: Instead of starting from Unicode characters, start from raw bytes (256 possible values). This guarantees that any input can be tokenized — even binary data or unusual Unicode. GPT-2, GPT-4, and LLaMA all use byte-level BPE.

- **Pre-tokenization**: Before applying BPE, split the text into chunks using regex patterns (e.g., separate words, numbers, and punctuation). This prevents merges from crossing word boundaries in undesirable ways.

- **Special tokens**: Reserved tokens like `<|endoftext|>`, `<|pad|>`, etc. are added to the vocabulary for controlling model behavior.

- **Speed**: Our implementation is $O(M \cdot N)$ where $M$ is the number of merges and $N$ is the corpus length. Libraries like `tiktoken` (OpenAI) and `tokenizers` (Hugging Face) use optimized data structures and are orders of magnitude faster.

The conceptual algorithm is exactly what we wrote above — the production versions just do it faster and on much more data.

### Three-way comparison

In [ ]:
# Compare all three on the same sentence
test_sentence = "the division of labour in the general business of society"

char_enc = char_tok.encode(test_sentence)
word_enc = word_tok.encode(test_sentence)
bpe_enc = bpe_encode(test_sentence, merges)

print(f"Sentence: '{test_sentence}'\n")
print(f"Character ({len(char_enc)} tokens):")
print(f"  {[char_tok.id_to_char[i] for i in char_enc]}\n")
print(f"Word ({len(word_enc)} tokens):")
print(f"  {[word_tok.id_to_token[i] for i in word_enc]}\n")
print(f"BPE-200 ({len(bpe_enc)} tokens):")
print(f"  {bpe_enc}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

methods = ["Character", "BPE-200", "Word"]
v_sizes = [char_tok.vocab_size, len(vocab), word_tok.vocab_size]
s_lens = [len(char_enc), len(bpe_enc), len(word_enc)]

x = np.arange(len(methods))
width = 0.35

bars1 = ax.bar(x - width / 2, v_sizes, width, label="Vocab size",
               color="steelblue", edgecolor="black", linewidth=0.5)
bars2 = ax.bar(x + width / 2, s_lens, width, label="Tokens in sentence",
               color="darkorange", edgecolor="black", linewidth=0.5)

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{int(bar.get_height())}", ha="center", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylabel("Count")
ax.set_title("Tokenization trade-offs")
ax.legend()
plt.tight_layout()
plt.show()

BPE occupies the sweet spot: a moderately sized vocabulary that produces reasonably short sequences, with no out-of-vocabulary problem.

### Tokenization and Embeddings

Once we have a tokenizer that maps text to integer IDs, we use `nn.Embedding` (which we studied in L08) to convert each ID into a learned dense vector. This is the input to the transformer.

The full pipeline:

$$\text{raw text} \xrightarrow{\text{tokenizer}} \text{token IDs} \in \mathbb{N}^T \xrightarrow{\texttt{nn.Embedding}} \text{dense vectors} \in \mathbb{R}^{T \times d}$$

where $T$ is the sequence length and $d$ is the embedding dimension.

In [ ]:
# Build a token-to-id mapping for our BPE vocabulary
bpe_token_to_id = {tok: i for i, tok in enumerate(vocab)}
bpe_vocab_size = len(vocab)

# Encode a sentence to integer IDs
sample = "the division of labour"
bpe_tokens = bpe_encode(sample, merges)
token_ids = torch.tensor([bpe_token_to_id[t] for t in bpe_tokens])

# Pass through an embedding layer
d_model = 64
embedding = nn.Embedding(bpe_vocab_size, d_model)
embedded = embedding(token_ids)

print(f"Text:          '{sample}'")
print(f"BPE tokens:    {bpe_tokens}")
print(f"Token IDs:     {token_ids.tolist()}")
print(f"ID shape:      {token_ids.shape}")
print(f"Embedded shape: {embedded.shape}  (seq_len x d_model)")

Each token ID has been mapped to a 64-dimensional vector. These vectors are the input to the transformer — which is the subject of the next notebook.

## References

- Gage, P. (1994). A new algorithm for data compression. *C Users Journal*, 12(2), 23–38.
- Sennrich, R., Haddow, B., & Birch, A. (2016). Neural machine translation of rare words with subword units. *ACL 2016*.
- Kudo, T., & Richardson, J. (2018). SentencePiece: A simple and language independent subword tokenizer and detokenizer for neural text processing. *EMNLP 2018*.
- Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). Language models are unsupervised multitask learners. *OpenAI Technical Report*.